CSV and Excel Files Parsing

In [1]:
import pandas as pd 
import os 

In [2]:
os.makedirs("data/structured_files", exist_ok=True)

In [3]:
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    'Stock': [50, 200, 150, 75, 100],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}
df = pd.DataFrame(data)
csv_path = "data/structured_files/products.csv"
df.to_csv(csv_path, index=False)
print(f"Sample CSV file created at: {csv_path}")


Sample CSV file created at: data/structured_files/products.csv


In [5]:
with pd.ExcelWriter("data/structured_files/products.xlsx") as writer:
    df.to_excel(writer, sheet_name="Products", index=False)
    summary_data={
        'Category': ['Electronics', 'Accessories'],
        'Total_Items':[3,2],
        'Total_Value':[1389.97,109.98]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)
print("Sample Excel file created at: data/structured_files/products.xlsx")



Sample Excel file created at: data/structured_files/products.xlsx


### CSV Processing

In [6]:
from langchain_community.document_loaders import CSVLoader,UnstructuredCSVLoader 

#### CSV Loader

In [7]:
csv_loader=CSVLoader(file_path="data/structured_files/products.csv", encoding="utf-8",csv_args={"delimiter":",","quotechar":'"',})
csv_docs=csv_loader.load()
print(f"Loaded {len(csv_docs)} documents using CSVLoader.")
for i,doc in enumerate(csv_docs):
    print(f"\nDocument {i+1}:")
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)
    

Loaded 5 documents using CSVLoader.

Document 1:
Content: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/structured_files/products.csv', 'row': 0}

Document 2:
Content: Product: Mouse
Category: Accessories
Price: 29.99
Stock: 200
Description: Wireless optical mouse with ergonomic design
Metadata: {'source': 'data/structured_files/products.csv', 'row': 1}

Document 3:
Content: Product: Keyboard
Category: Accessories
Price: 79.99
Stock: 150
Description: Mechanical keyboard with RGB backlighting
Metadata: {'source': 'data/structured_files/products.csv', 'row': 2}

Document 4:
Content: Product: Monitor
Category: Electronics
Price: 299.99
Stock: 75
Description: 27-inch 4K monitor with HDR support
Metadata: {'source': 'data/structured_files/products.csv', 'row': 3}

Document 5:
Content: Product: Webcam
Category: Electronics
Price: 89.99
Stock: 100
Description: 1080p webcam with noise ca

#### Custom CSV Processing

In [9]:
from langchain_core.documents import Document
def process_csv_intelligently(file_path)->list[Document]:
    try:
        df = pd.read_csv(file_path)
        documents = []
        for idx, row in df.iterrows():
            content=f"""Product Information:
            Name: {row['Product']}
            Category: {row['Category']}
            Price: ${row['Price']}
            Stock: {row['Stock']}
            Description: {row['Description']}"""
            metadata = {
                'source': file_path,
                'row_index': idx,
                'product_name': row['Product'],
                'category': row['Category'],
                'price': row['Price'],
                'stock': row['Stock'],
                "data_type": "product_info"
            }
            documents.append(Document(page_content=content, metadata=metadata))
        return documents
    except Exception as e:
        print(f"Error processing CSV file: {e}")
        return []

In [10]:
processed_csv_docs=process_csv_intelligently("data/structured_files/products.csv")
print(f"\nProcessed {len(processed_csv_docs)} documents from CSV file.")
for i,doc in enumerate(processed_csv_docs):
    print(f"\nDocument {i+1}:")
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)
    


Processed 5 documents from CSV file.

Document 1:
Content: Product Information:
            Name: Laptop
            Category: Electronics
            Price: $999.99
            Stock: 50
            Description: High-performance laptop with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'stock': 50, 'data_type': 'product_info'}

Document 2:
Content: Product Information:
            Name: Mouse
            Category: Accessories
            Price: $29.99
            Stock: 200
            Description: Wireless optical mouse with ergonomic design
Metadata: {'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'stock': 200, 'data_type': 'product_info'}

Document 3:
Content: Product Information:
            Name: Keyboard
            Category: Accessories
            Price: $79.99
           

## Excel Processing

In [11]:
def process_excel_with_pandas(filepath)->list[Document]:
    documents=[]
    excel_file=pd.ExcelFile(filepath)
    for sheet_name in excel_file.sheet_names:
        df=pd.read_excel(excel_file, sheet_name=sheet_name)
        sheet_column=f"Sheet: {sheet_name}"
        sheet_column+=f"Columns: {', '.join(df.columns)}"
        sheet_column+=f"Rows: {len(df)}"
        sheet_column+=df.to_string(index=False)
        doc = Document(page_content=sheet_column, metadata={"source": filepath, "sheet_name": sheet_name,
        "num_rows": len(df), "num_columns": len(df.columns),"data_type": "excel_sheet"})
        documents.append(doc)
    return documents

In [12]:
excel_docs=process_excel_with_pandas("data/structured_files/products.xlsx")
print(f"\nProcessed {len(excel_docs)} documents from Excel file.")
for i,doc in enumerate(excel_docs):
    print(f"\nDocument {i+1}:")
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)
    


Processed 2 documents from Excel file.

Document 1:
Content: Sheet: ProductsColumns: Product, Category, Price, Stock, DescriptionRows: 5 Product    Category  Price  Stock                                         Description
  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD
   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design
Keyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting
 Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support
  Webcam Electronics  89.99    100                1080p webcam with noise cancellation
Metadata: {'source': 'data/structured_files/products.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}

Document 2:
Content: Sheet: SummaryColumns: Category, Total_Items, Total_ValueRows: 2   Category  Total_Items  Total_Value
Electronics            3      1389.97
Accessories            2       1

In [14]:
from langchain_community.document_loaders import UnstructuredExcelLoader
try:
    excel_loader=UnstructuredExcelLoader(file_path="data/structured_files/products.xlsx", mode="elements")
    excel_docs_unstructured=excel_loader.load()
    print(f"\nLoaded {len(excel_docs_unstructured)} documents using UnstructuredExcelLoader.")
    for i,doc in enumerate(excel_docs_unstructured):
        print(f"\nDocument {i+1}:")
        print("Content:", doc.page_content)
        print("Metadata:", doc.metadata)
except Exception as e:
    print(f"Error loading Excel file with UnstructuredExcelLoader: {e}")



Loaded 2 documents using UnstructuredExcelLoader.

Document 1:
Content: Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SSD Mouse Accessories 29.99 200 Wireless optical mouse with ergonomic design Keyboard Accessories 79.99 150 Mechanical keyboard with RGB backlighting Monitor Electronics 299.99 75 27-inch 4K monitor with HDR support Webcam Electronics 89.99 100 1080p webcam with noise cancellation
Metadata: {'source': 'data/structured_files/products.xlsx', 'file_directory': 'data/structured_files', 'filename': 'products.xlsx', 'last_modified': '2026-04-24T00:23:21', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product</td><td>Category</td><td>Price</td><td>Stock</td><td>Description</td></tr><tr><td>Laptop</td><td>Electronics</td><td>999.99</td><td>50</td><td>High-performance laptop with 16GB RAM and 512GB SSD</td></tr><tr><td>Mouse</td><td>Accessories</td><td>29.99</td><td>200</td><td>